In [1]:
file1 = "twitter_scraping_data_20260109_071513.parquet"
file2 = "twitter_scraping_data_20260109_191512.parquet"

In [2]:
import pandas as pd
import re

In [3]:
df_1 = pd.read_parquet(file1)
df_2 = pd.read_parquet(file2)

In [4]:
df = pd.concat([df_2, df_1], ignore_index=True)

In [5]:
df = df.drop_duplicates(subset=['post_url', 'author'], keep='first').copy()

In [6]:
df.head()

,author,post_url,content,date,replies,retweets,likes,bookmarks,views,topic,supported_industry,is_tech_related
0,karpathy,https://x.com/karpathy/status/2009037707918626874,New post: nanochat miniseries v1\n\nThe correc...,2026-01-07T23:01:30.000Z,174,564,4638,3807,471914,[Natural Language Processing (NLP)],[Technology / AI],True
1,karpathy,https://x.com/karpathy/status/2008664551445963083,The majority of the ruff ruff is people who lo...,2026-01-06T22:18:42.000Z,272,301,4248,626,343346,[Non-technical],[],False
2,rglucks1,https://x.com/rglucks1/status/2009188297290207248,"Dear American friends,\n\nWhen will you wake u...",2026-01-08T08:59:53.000Z,678,980,5902,158,590747,[Non-technical],[],False
3,AC360,https://x.com/AC360/status/2008724649983500325,The White House Website is officially re-writi...,2026-01-07T02:17:31.000Z,322,2148,5022,387,138285,[Non-technical],[],False
4,AndrewYNg,https://x.com/AndrewYNg/status/200857874131283...,Happy 2026! Will this be the year we finally a...,2026-01-06T16:37:44.000Z,165,302,1490,553,142652,[Non-technical],[],False


In [160]:
pip install spacy


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 23.6 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 KB 18.3 MB/s eta 0:00:0031m16.9 MB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 KB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.2 MB/s eta 0:00:000:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.7/229.7 KB 15.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 KB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 KB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 28.1 MB/s eta 0:00:00m eta 0:00:010:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 27.0 MB/s eta 0:00:00m eta 0:00:010:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 KB 19.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━

In [167]:
!python3 -m spacy download en_core_web_sm


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 9.5 MB/s eta 0:00:000m eta 0:00:01:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [7]:
import spacy
from itertools import chain

nlp = spacy.load("en_core_web_sm")

# ---- optional curated synonym map (HIGH PRECISION ONLY) ----
TECH_SYNONYMS = {
    "optimizer": {"adam", "sgd", "rmsprop"},
    "loss": {"loss function", "objective"},
    "embedding": {"vector embedding"},
    "transformer": {"attention model"},
    "pipeline": {"workflow"},
    "deployment": {"serving"},
    "tokenization": {"tokeniser"},
    "hyperparameter": {"tunable parameter"},
}

def normalize_phrase(text: str) -> str:
    return text.lower().replace("-", " ").strip()

def lemmatize_phrase(text: str) -> str:
    doc = nlp(text)
    return " ".join(tok.lemma_ for tok in doc if not tok.is_stop)

def enrich_technical_keywords(seed_keywords: set[str]) -> set[str]:
    enriched = set()

    for kw in seed_keywords:
        kw = normalize_phrase(kw)
        enriched.add(kw)

        # lemma form
        lemma = lemmatize_phrase(kw)
        if lemma:
            enriched.add(lemma)

        # underscore n-gram version
        if " " in kw:
            enriched.add(kw.replace(" ", "_"))

        # curated synonyms only
        if kw in TECH_SYNONYMS:
            for syn in TECH_SYNONYMS[kw]:
                enriched.add(normalize_phrase(syn))

    return enriched


In [7]:
BINARY_LABELS = [
    (
        "Technical content about AI, data, machine learning, NLP, or software engineering "
        "implementation details such as methods, architectures, algorithms, "
        "or code-level discussion"
    ),
    (
        "Non-technical content such as opinions, social or political commentary, "
        "industry news, announcements, founder stories, product launches, "
        "hiring posts, or high-level discussion of AI without technical details"
    )
]

LABEL_DEFINITIONS = {
    (
        "Technical content on generative AI and foundation models, including large language models (LLMs), "
        "agent-based systems, prompt engineering, pretraining and fine-tuning, "
        "evaluation methodologies, and comparative analysis of generative models."
    ): "Generative AI",
    (
        "Natural Language Processing (NLP), focusing on technical methods and systems for understanding, analyzing, "
        "and transforming human language text or speech. Includes tasks such as tokenization, parsing, "
        "named entity recognition, text classification, information extraction, embeddings, "
        "semantic similarity, and linguistic feature modeling. Excludes high-level commentary or opinions."
    ): "NLP",
    (
        "Machine Learning, focusing on technical content with details of general machine learning algorithms, theory, or training methods "
        "such as supervised, unsupervised, or reinforcement learning that are not specific "
        "to generative foundation models."
    ): "Machine Learning",

    (
        "MLOps and Orchestration, focusing on operational systems for machine learning, "
        "including training pipelines, deployment, monitoring, CI/CD, infrastructure, "
        "and reliability engineering. Excludes model research, scaling laws, or training science."
    ): "Orchestration",

    (
        "Data Analytics, focusing on data analysis, reporting, dashboards, statistics, "
        "and business intelligence rather than machine learning model development."
    ): "Data Analytics",

    (
        "Robotics, focusing on AI systems for robotic perception, control, motion planning, "
        "and automation in physical or autonomous robotic systems."
    ): "Robotics"
}



In [8]:

TECHNICAL_KEYWORDS = {
    # Core ML / AI (high signal)
    "gradient", "loss", "optimizer", "backpropagation",
    "overfitting", "underfitting", "regularization",
    "hyperparameter", "learning rate", "convergence",
    "objective function", "cost function", "ML", "AI",
    "engineering", "software", "deep learning", "engineer", "research",

    # Model lifecycle
    "pretraining", "fine-tuning", "instruction tuning",
    "supervised learning", "unsupervised learning",
    "reinforcement learning", "rlhf",
    "transfer learning", "zero-shot", "few-shot",

    # LLM / GenAI internals
    "transformer", "self-attention", "cross-attention",
    "tokenization", "subword", "byte-pair encoding",
    "positional encoding", "context window",
    "logits", "softmax", "sampling",
    "temperature", "top-k", "top-p",
    "hallucination", "prompt injection", "agent", "agents",
    "llm", "llms",

    # Retrieval / RAG
    "embedding", "vector", "vector store",
    "semantic search", "nearest neighbor",
    "indexing", "faiss", "hnsw",
    "retrieval augmented generation",

    # NLP (classic + modern)
    "named entity recognition", "ner",
    "part-of-speech tagging", "pos tagging",
    "dependency parsing", "syntactic parsing",
    "language modeling", "text classification",
    "sequence labeling",

    # Evaluation / experimentation
    "benchmark", "evaluation metric",
    "precision", "recall", "f1 score",
    "bleu", "rouge", "perplexity",
    "ablation study", "baseline",

    # Systems / infra
    "pipeline", "data pipeline", "data",
    "training pipeline", "inference pipeline",
    "throughput", "latency", "batching",
    "streaming", "autoscaling",
    "load balancing", "fault tolerance",

    # MLOps / deployment
    "deployment", "serving",
    "model serving", "model registry",
    "versioning", "monitoring",
    "drift detection", "feature store",
    "ci/cd", "canary deployment",

    # Engineering / tooling
    "api", "sdk", "endpoint",
    "rest", "grpc",
    "container", "docker",
    "kubernetes", "helm",
    "python", "pytorch", "tensorflow",
    "jax", "onnx",

    # Rule-based / symbolic systems
    "decision tree", "finite-state machine",
    "rule-based system", "if-then rules",
    "symbolic ai", "expert system",
    
    # Core robotics
    "robotics", "robot", "autonomous system",
    "manipulator", "mobile robot", "humanoid",
    "end effector", "degrees of freedom", "dof"
}

ops_keywords = [
        # MLOps
        'mlops', 'ml ops', 'mlflow', 'kubeflow', 'airflow', 'prefect',
        'model deployment', 'model serving', 'model monitoring',
        'model registry', 'feature store', 'experiment tracking',
        'model versioning', 'model lifecycle', 'model pipeline',

        # LLMOps
        'llmops', 'llm ops', 'langsmith', 'langfuse', 'helicone',
        'weights & biases', 'wandb',
        'prompt management', 'prompt versioning',
        'llm monitoring', 'llm observability',
        'token usage', 'inference cost',
        'llm deployment', 'llm serving', 'model endpoint',
        'context window', 'prompt caching', 'llm gateway',
        'openai api', 'anthropic api', 'llm evaluation',

        # AgentOps
        'agentops', 'agent ops', 'agent monitoring', 'agent orchestration',
        'agent deployment', 'agent observability',
        'langgraph', 'crewai', 'autogen',
        'agent tracing', 'agent logging',
        'multi-agent system', 'agent workflow',
        'tool calling', 'function calling',
        'agent retry', 'agent state',

        # DevOps / CI
        'devops', 'ci/cd', 'continuous integration',
        'continuous deployment', 'github actions',
        'gitlab ci', 'jenkins', 'workflow yaml',
        'gh cli', 'github cli', 'ci failure', 'build failure',

        # Infra
        'kubernetes', 'k8s', 'docker', 'container',
        'terraform', 'ansible', 'infrastructure as code',

        # Cloud
        'aws', 'sagemaker', 'azure ml', 'vertex ai',
        'lambda', 'ec2', 'eks', 'ecs', 'cloud deployment',

        # Observability
        'monitoring', 'observability', 'logging',
        'metrics', 'tracing', 'alerting',
        'latency', 'throughput', 'reliability',

        # Data infra
        'data pipeline', 'etl', 'data orchestration',
        'kafka', 'spark', 'dask'
    ]


In [9]:
def is_ops_mlops_rule_based(text):
    """
    Rule-based filter to identify Ops / MLOps / LLMOps / AgentOps content.
    Returns True if post is about operations, infrastructure, deployment,
    CI/CD, debugging, monitoring, or production systems.
    """
    print(" --> MLOps rule bases")
    
    if not isinstance(text, str) or not text.strip():
        return False

    text_lower = text.lower()

   
    ops_score = sum(1 for kw in ops_keywords if kw in text_lower)

    # --------------------------------------------------
    # Ops / deployment / CI phrases
    # --------------------------------------------------
    ops_phrases = [
        r'\b(deploy|debug|investigate|fix|analyze)\b.*\b(ci|pipeline|workflow|build)\b',
        r'\b(ci|pipeline|workflow)\b.*\b(fail|failure|broken|stuck)\b',
        r'\b(deploy(ing|ed|ment)?|serving|host(ing)?)\s+(model|ml|llm|agent)\b',
        r'\b(model|llm|agent)\s+(deployment|serving|monitoring|pipeline)\b',
        r'\b(container(ize|ized)?|docker(ize|ized)?)\b',
        r'\b(orchestrat(e|ing|ion)|automat(e|ing|ion))\b',
        r'\b(kubernetes|k8s)\b',
        r'\b(prompt|token|inference)\s+(cost|usage|monitoring|caching)\b',
        r'\b(agent)\s+(orchestration|deployment|monitoring|workflow)\b',
        r'\b(langchain|langgraph|autogen|crewai)\b.*\b(deploy|production)\b'
    ]

    phrase_matches = sum(1 for p in ops_phrases if re.search(p, text_lower))

    # --------------------------------------------------
    # Short imperative ops tasks (important for X)
    # --------------------------------------------------
    ops_verbs = [
        'debug', 'deploy', 'monitor', 'fix', 'investigate',
        'analyze', 'scale', 'configure', 'optimize'
    ]

    if len(text.split()) < 50:
        if any(v in text_lower for v in ops_verbs) and ops_score >= 1:
            return True

    # --------------------------------------------------
    # Strong indicators
    # --------------------------------------------------
    if ops_score >= 3 or phrase_matches >= 2:
        return True

    # Medium indicators + operational context
    if ops_score >= 2 or phrase_matches >= 1:
        operational_context = [
            'production', 'pipeline', 'infrastructure',
            'latency', 'performance', 'availability',
            'reliability', 'scale'
        ]

        context_score = sum(1 for k in operational_context if k in text_lower)
        if context_score >= 1:
            return True

    return False


def is_non_technical_rule_based(text):
    """
    Returns True if post is clearly non-technical.
    Conservative filter: only returns False if strong technical evidence exists.
    """
    print(" --> Classification technical rule bases")
    if not isinstance(text, str) or not text.strip():
        return True

    text_lower = text.lower()

    if any(kw in text_lower for kw in TECHNICAL_KEYWORDS):
        return False

    return True

def is_empty(text):
    return not isinstance(text, str) or not text.strip()

def is_too_short(text, min_words=25):
    return len(text.split()) < min_words

In [10]:
# -------- zero_shot ---------
from transformers import pipeline

binary_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

multi_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

def is_non_tech_zero_shot(
    text,
    tech_threshold=0.52
):
    """
    Strong bias toward Non-technical.
    Returns True if Non-technical, False if Technical.
    """
    print(" --> Binary classification by zero shot")
    result = binary_classifier(
        text,
        BINARY_LABELS,
        multi_label=False
    )
    scores = dict(zip(result["labels"], result["scores"]))
    tech_label, _ = BINARY_LABELS
    tech_score = scores.get(tech_label, 0.0)
    
    if tech_score < tech_threshold:
        return True

    return False

expanded_labels = list(LABEL_DEFINITIONS.keys())
def classify_zero_shot(text, margin=0.06):

    print(" --> Zero shot tech classifcation")
    result = multi_classifier(
        text,
        expanded_labels,
        multi_label=True
    )

    scores = {
        LABEL_DEFINITIONS[label]: score
        for label, score in zip(result["labels"], result["scores"])
    }

    return scores


In [11]:
def classify_text(text, margin=0.06):
    if is_empty(text) or is_too_short(text):
        print(" --> Text is not long enough")
        return ["Non-technical"]

    if is_non_technical_rule_based(text):
        return ["Non-technical"]

    if is_non_tech_zero_shot(text):
        return ["Non-technical"]
    res = []
    if (is_ops_mlops_rule_based(text)):
        res.append("Orchestration")

    scores = classify_zero_shot(text)
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    top_label, top_score = sorted_scores[0]
    second_label, second_score = (
        sorted_scores[1] if len(sorted_scores) > 1 else (None, 0.0)
    )

    res = [top_label]
    if second_label and abs(top_score - second_score) <= margin:
        res.append(second_label)

    return res


In [12]:
def pd_to_text(df, post_url):
    return df[df.post_url == post_url].content.values[0]

In [13]:
text = ("New quantum method lets drones, #Robots talk even in ‘signal lost’ zones"
"by Rupendra Brahambhatt" 
"@IntEngineering"
"Learn more: https://bit.ly/4auLlYE"
"#Robotics #Engineering #ArtificialIntelligence #Innovation #Technology"    
       )
print(text)
classify_text(text)

New quantum method lets drones, #Robots talk even in ‘signal lost’ zonesby Rupendra Brahambhatt@IntEngineeringLearn more: https://bit.ly/4auLlYE#Robotics #Engineering #ArtificialIntelligence #Innovation #Technology
 --> Text is not long enough


['Non-technical']

In [14]:
text = (
"I've solved a second Erdos problem (#281) using only GPT 5.2 Pro - no prior solutions found."

"Terence Tao calls it 'perhaps the most unambiguous instance' of AI solving an open problem:"
)
classify_text(text)

 --> Classification technical rule bases


['Non-technical']

In [64]:
def pd_to_text(df, post_url):
    return df[df.post_url == post_url].content.values[0]

In [65]:
text = (
"New quantum method lets drones, #Robots talk even in ‘signal lost’ zones"
"by Rupendra Brahambhatt" 
"@IntEngineering"
"Learn more: https://bit.ly/4auLlYE"
"#Robotics #Engineering #ArtificialIntelligence #Innovation #Technology"    
       )
print(text)
classify_text(text)

New quantum method lets drones, #Robots talk even in ‘signal lost’ zonesby Rupendra Brahambhatt@IntEngineeringLearn more: https://bit.ly/4auLlYE#Robotics #Engineering #ArtificialIntelligence #Innovation #Technology
 --> Text is not long enough


['Non-technical']

In [78]:
text = ("ChatGPT 5.2 Pro provides an elegant proof of the explosive growth result in Example 3 (pp. 256-257) in Aghion, Jones, and Jones (2019)   1/N")
print(text)
classify_text(text)

ChatGPT 5.2 Pro provides an elegant proof of the explosive growth result in Example 3 (pp. 256-257) in Aghion, Jones, and Jones (2019)   1/N
 --> Text is not long enough


['Non-technical']

In [79]:
text = (
"Random normies talking about “I asked gpt” “I vibe-coded with Claude” and you still think early AI art isn’t valuable"

"They’re Paleolithic artifacts and this is completely obvious if you have any foresight whatsoever"
)
classify_text(text)

 --> Classification technical rule bases


['Non-technical']

In [80]:
text = pd_to_text(df, "https://x.com/HamelHusain/status/2009506396061196666")
print(text)
classify_text(text)

One of AI's best uses is to get you out of GH Actions hell

> Look at why CI is failing using gh cli, and figure out why its failing.  Propose a fix.
 --> Classification technical rule bases


['Non-technical']

In [81]:
text = pd_to_text(df, "https://x.com/AMD/status/2009434470290526620")
print(text)
classify_text(text)

Scale big...
@TomsHardware
 spotlights AMD Helios rack scale AI architecture and Instinct MI430X/MI440X/MI455X – an MI400 series built to meet diverse infrastructure and customer needs. 

Get the details: 
tomshardware.com/tech-industry/
…
 --> Classification technical rule bases


['Non-technical']

In [82]:
text = pd_to_text(df, "https://x.com/AINowInstitute/status/1995945610646880320")
print(text)

classify_text(text)

We’re expanding our national security and defense work, and welcoming Boyan Milanov to the team. 
@bmilanov
 is a research scientist evaluating cybersecurity risks in agentic AI systems related to national security, defense & safety-critical infrastructure. 
ainowinstitute.org/contributor/bo
…
 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'We’re expanding our national security and defense work, and welcoming Boyan Milanov to the team. \n@bmilanov\n is a research scientist evaluating cybersecurity risks in agentic AI systems related to national security, defense & safety-critical infrastructure. \nainowinstitute.org/contributor/bo\n…', 'labels': ['Non-technical content such as opinions, social or political commentary, industry news, announcements, founder stories, product launches, hiring posts, or high-level discussion of AI without technical details', 'Technical content about AI, data, machine learning, NLP, or software engineering imple

['Non-technical']

In [83]:
text = pd_to_text(df, "https://x.com/alexolegimas/status/2007818641526403302")
print(text)

classify_text(text)

When ChatGPT came out a few years ago I was caught completely off guard. My first reaction was anxiety and panic--how did I miss this technology that was going to change the world. 

Luckily I had some friends like 
@m_sendhil
 and 
@alex_peys
 who had been on the forefront for years to give me advice. And the advice was exactly the same: to understand it, just train your own model from scratch.

That's what I did. I think this has helped me tremendously, both in seeing the promise and understand the (potential) limits of the tech. 

There's a lot of excellent youtube resources on this, including by 
@karpathy
. But if you're interested in doing the same, I've recently gone through 
@rasbt
's "Build a Large Language Model" and found it to be the best option.
 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'When ChatGPT came out a few years ago I was caught completely off guard. My first reaction was anxiety and panic--how did I miss this te

['Non-technical']

In [84]:
text = pd_to_text(df, "https://x.com/bhalligan/status/2007130064056439234")
print(text)
classify_text(text)

I’m starting to worry about Massachusetts 
1. Biotech is way off from a few years ago
2. Only 1 of the top 50 ai companies are in MA
3. The Fed research funding cuts hitting MIT, Harvard, Whoi are brutal.
4. The millionaires tax is working in the short run, but I know a lot of wealthy folks preparing for a FL move.
5. A glut of empty condos
6.  It’s not “cool” for young folks
7.  It’s expensive as sh-t.

I honestly don’t think the MA/Boston govt can do that much about it as they are kind of macro issues.  I give them big credit for working on building more housing and fixing the T, which will help.

I’m trying to help w HubSpot, partnering w WHOI, teaching at MIT.  I’d like to help more.  Specifically I’d like to encourage and help more ai and climate companies in the state.  I think ai and climate should be our dual growth engines.
 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'I’m starting to worry about Massachusetts \n1. Biotech is wa

['Non-technical']

In [85]:
text = "Exchange vLLM recipes, share memory compression techniques, discuss frontier coding models, and more in our Generative AI study group! Join us every Friday at 8 am PT by registering at https://twimlai.com/community/. See you there!"
print(text)
classify_text(text)

Exchange vLLM recipes, share memory compression techniques, discuss frontier coding models, and more in our Generative AI study group! Join us every Friday at 8 am PT by registering at https://twimlai.com/community/. See you there!
 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'Exchange vLLM recipes, share memory compression techniques, discuss frontier coding models, and more in our Generative AI study group! Join us every Friday at 8 am PT by registering at https://twimlai.com/community/. See you there!', 'labels': ['Technical content about AI, data, machine learning, NLP, or software engineering implementation details such as methods, architectures, algorithms, or code-level discussion', 'Non-technical content such as opinions, social or political commentary, industry news, announcements, founder stories, product launches, hiring posts, or high-level discussion of AI without technical details'], 'scores': [0.9075721502304077, 0.0924278

['Generative AI']

In [86]:
text = "Join our Generative AI study group tomorrow at 8 am PT and share feature engineering strategies, explore SLMs for agents, discuss hybrid retrieval architectures, and more! If you haven’t registered yet, head over to https://twimlai.com/community/ to sign up."
print(text)
classify_text(text)

Join our Generative AI study group tomorrow at 8 am PT and share feature engineering strategies, explore SLMs for agents, discuss hybrid retrieval architectures, and more! If you haven’t registered yet, head over to https://twimlai.com/community/ to sign up.
 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'Join our Generative AI study group tomorrow at 8 am PT and share feature engineering strategies, explore SLMs for agents, discuss hybrid retrieval architectures, and more! If you haven’t registered yet, head over to https://twimlai.com/community/ to sign up.', 'labels': ['Technical content about AI, data, machine learning, NLP, or software engineering implementation details such as methods, architectures, algorithms, or code-level discussion', 'Non-technical content such as opinions, social or political commentary, industry news, announcements, founder stories, product launches, hiring posts, or high-level discussion of AI without technic

['Generative AI']

In [87]:
ai_text = """
Artificial Intelligence systems that generate text are fundamentally built on large-scale data processing,
statistical learning, and advanced neural network architectures. At the core of modern AI-to-text systems
are deep learning models, most notably transformer-based architectures, which are designed to capture
long-range dependencies in language data.

These models are trained on massive corpora of text that include books, articles, code repositories,
and structured technical documents. During training, the model learns probabilistic relationships
between tokens, enabling it to predict the next token in a sequence with high accuracy.

Technical data plays a crucial role in improving the reliability and usefulness of AI-generated text.
Structured datasets such as logs, tables, telemetry data, and knowledge graphs are often transformed
into machine-readable formats before being integrated into training or inference pipelines.

Feature engineering, data normalization, and tokenization strategies are applied to ensure that both
numerical and textual information can be effectively represented within the model’s embedding space.
This allows AI systems to generate text that reflects technical precision, domain-specific terminology,
and contextual consistency.

Large language models rely heavily on distributed computing infrastructure. Training typically involves
thousands of GPUs or specialized accelerators operating in parallel, coordinated through data and model
parallelism techniques. Optimization algorithms such as Adam or RMSProp are used to minimize loss
functions, while techniques like gradient clipping and learning rate scheduling help stabilize training
at scale.

AI-to-text systems are increasingly integrated into real-world technical workflows. They are used to
generate documentation, summarize technical logs, translate structured data into human-readable reports,
and assist in software development by producing code explanations or configuration templates.

As the field evolves, researchers are exploring ways to make AI text generation more transparent and
controllable. Techniques such as fine-tuning on domain-specific datasets, reinforcement learning from
human feedback, and retrieval-augmented generation allow models to align more closely with technical
requirements and up-to-date information.

Overall, AI-driven text generation represents a convergence of data engineering, machine learning,
and systems design. Its effectiveness depends on the careful integration of high-quality technical data,
scalable infrastructure, and rigorous evaluation methods.
"""
classify_text(ai_text)

 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': '\nArtificial Intelligence systems that generate text are fundamentally built on large-scale data processing,\nstatistical learning, and advanced neural network architectures. At the core of modern AI-to-text systems\nare deep learning models, most notably transformer-based architectures, which are designed to capture\nlong-range dependencies in language data.\n\nThese models are trained on massive corpora of text that include books, articles, code repositories,\nand structured technical documents. During training, the model learns probabilistic relationships\nbetween tokens, enabling it to predict the next token in a sequence with high accuracy.\n\nTechnical data plays a crucial role in improving the reliability and usefulness of AI-generated text.\nStructured datasets such as logs, tables, telemetry data, and knowledge graphs are often transformed\ninto machine-readable formats before being 

['Machine Learning']

In [88]:
text = "We are looking for a skilled AI / Machine Learning Engineer to join our growing team and help build intelligent, data-driven systems at scale. This role is ideal for someone who enjoys working on real-world ML problems, from model development to production deployment."
classify_text(text)

 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'We are looking for a skilled AI / Machine Learning Engineer to join our growing team and help build intelligent, data-driven systems at scale. This role is ideal for someone who enjoys working on real-world ML problems, from model development to production deployment.', 'labels': ['Non-technical content such as opinions, social or political commentary, industry news, announcements, founder stories, product launches, hiring posts, or high-level discussion of AI without technical details', 'Technical content about AI, data, machine learning, NLP, or software engineering implementation details such as methods, architectures, algorithms, or code-level discussion'], 'scores': [0.5617656707763672, 0.4382343292236328]}


['Non-technical']

In [89]:
text = pd_to_text(df, "https://x.com/STLChrisH/status/2008924838669295685")
print(text)
classify_text(text)

Favorite excerpt from Brent Beshore's annual letter: 

What CEOs Are and Aren’t

Most people think of a CEO as the person at the top. That’s true in the same way it’s true that the windshield is “at the front” of the car. Technically correct. 

Also, misses the point. The windshield isn’t the engine. It isn’t the wheels. It doesn’t move anything. But it does determine what the driver can see, what they ignore, and what they slam into at 70 miles an hour.

When done well, the CEO job is an arbiter of truth. The CEO stands at the border between the outside world and the inside world, between company mythology and competitive reality. That sounds obvious, but it’s not. 

I’d argue the norm is delusion, where organizations create realities disconnected from truth, complete with alternate headlines, villains, and heroes, all proclaimed with a shocking level of certainty.

So the CEO’s job starts with a basic question: What’s true?

Not what’s comforting. Not what’s politically convenient. N

['Non-technical']

In [90]:
text = ("Rules-based NLP "
"The earliest NLP applications were simple if-then decision trees, requiring preprogrammed rules. They are only able to provide answers in response to specific prompts, such as the original version of Moviefone, which had rudimentary natural language generation (NLG) capabilities. Because there is no machine learning or AI capability in rules-based NLP, this function is highly limited and not scalable."
       )
is_non_tech_zero_shot(text)


 --> Binary classification by zero shot
{'sequence': 'Rules-based NLP The earliest NLP applications were simple if-then decision trees, requiring preprogrammed rules. They are only able to provide answers in response to specific prompts, such as the original version of Moviefone, which had rudimentary natural language generation (NLG) capabilities. Because there is no machine learning or AI capability in rules-based NLP, this function is highly limited and not scalable.', 'labels': ['Technical content about AI, data, machine learning, NLP, or software engineering implementation details such as methods, architectures, algorithms, or code-level discussion', 'Non-technical content such as opinions, social or political commentary, industry news, announcements, founder stories, product launches, hiring posts, or high-level discussion of AI without technical details'], 'scores': [0.5842839479446411, 0.41571611166000366]}


False

In [91]:
text = pd_to_text(df, "https://x.com/rasbt/status/2007122635507880251")
print(text)
classify_text(text)

Another really interesting paper from my 2025 bookmarked papers: On the Interplay of Pre-Training, Mid-Training, and RL on Reasoning Language Models (
arxiv.org/abs/2512.07783).

In short, RL is most effective when applied to data that is neither too close to nor too far from the pre-training distribution. 

If the data is too in-distribution, RL adds little beyond supervised training. If it is too far out-of-distribution, RL struggles because the model lacks the necessary priors.

This has been known before, but it's nice to see it formalized with data and figures to reference.
 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': "Another really interesting paper from my 2025 bookmarked papers: On the Interplay of Pre-Training, Mid-Training, and RL on Reasoning Language Models (\narxiv.org/abs/2512.07783).\n\nIn short, RL is most effective when applied to data that is neither too close to nor too far from the pre-training distribution. \n\nIf t

['Generative AI']

In [92]:
text = pd_to_text(df, "https://x.com/AlexanderMcCoy4/status/2009353547285225569")
print(text)
classify_text(text)

I’m a Marine Corp veteran, trained by LEOs when I served in US embassies on how to handle unruly individuals or protestors.

1) If you are the guys with guns, you are the ones responsible for the situation. Doubly so if you outnumber the person you’re engaging.

2) You do not need to use force except to control the situation in order to deescalate it. Minimal force required.

3) You are NOT here to look for excuses to use more force. Even if the person gives you an excuse which “justifies” using force, that doesn’t mean using force is de facto the right move.

4) “Let the other person retreat” often resolves the situation just fine! Don’t surround people, back them up against a wall, etc. Your job is to control the situation. “I put myself stupidly in danger” is not an excuse to escalate “because I’m in danger.”

5) People will feed off of your energy. If you come rolling up like a fascist thug ready to break skulls, people will meet you at that level. If you show up calm, professional

['Non-technical']

In [93]:
text = ("Somewhere a quiet engineer closes their laptop."

"They walk out before the sky finishes its warning."
)


label = classify_text(text)
label

 --> Text is not long enough


['Non-technical']

In [94]:
text = (
    "Small-but-happy win: If you tell ChatGPT not to use em-dashes in your "
    "custom instructions, it finally does what it's supposed to do!"
)

label = classify_text(text)
label

 --> Text is not long enough


['Non-technical']

In [95]:
text = (
"I will be to share that I’m starting a new position as ML engineer at Advanced Machine Intelligence - AMI Labs!"
)
label = classify_text(text)
label

 --> Text is not long enough


['Non-technical']

In [98]:
text = "The launch of ChatGPT Health is really personal for me. I know how hard it can be to navigate the healthcare system (even with great care). AI can help patients and doctors with some of the biggest issues. More here: https://fidjisimo.substack.com/p/chatgpt-health"
label = classify_text(text)
label

 --> Classification technical rule bases


['Non-technical']

In [99]:
# expression
text = (
    "I love ChatGPT because it feels like talking to a really smart and patient friend. "
    "It helps me think through ideas, write better messages, and feel more confident when "
    "expressing myself. I’m always impressed by how friendly, supportive, and encouraging "
    "the responses are. It makes learning and creating feel easier and more enjoyable, "
    "even on days when I feel stuck or unmotivated."
)
label = classify_text(text)
label


 --> Classification technical rule bases


['Non-technical']

In [100]:
text = (
    "My 8,000-word note on agents: https://huyenchip.com/2025/01/07/agents.html\n\n"
    "Covering:\n\n"
    "1. An overview of agents\n"
    "2. How the capability of an AI-powered agent is determined by the set of tools it has access to "
    "and its capability for planning\n"
    "3. How to select the best set of tools for your agent\n"
    "4. Whether LLMs can plan and how to augment a model’s capability for planning\n"
    "5. Agent failure modes\n\n"
    "AI-powered agents are an emerging field with no established theoretical frameworks for defining, "
    "developing, and evaluating them. This post is a best-effort attempt to build a framework from the "
    "existing literature, but it will evolve as the field does.\n\n"
    "As always, feedback is much appreciated!"
)


label = classify_text(text)
label

 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'My 8,000-word note on agents: https://huyenchip.com/2025/01/07/agents.html\n\nCovering:\n\n1. An overview of agents\n2. How the capability of an AI-powered agent is determined by the set of tools it has access to and its capability for planning\n3. How to select the best set of tools for your agent\n4. Whether LLMs can plan and how to augment a model’s capability for planning\n5. Agent failure modes\n\nAI-powered agents are an emerging field with no established theoretical frameworks for defining, developing, and evaluating them. This post is a best-effort attempt to build a framework from the existing literature, but it will evolve as the field does.\n\nAs always, feedback is much appreciated!', 'labels': ['Technical content about AI, data, machine learning, NLP, or software engineering implementation details such as methods, architectures, algorithms, or code-level discussion', 'Non-technic

['Generative AI']

In [101]:
text = (
    "3. Dynamic few-shot prompting\n\n"
    "They cautioned against using traditional few-shot prompting for agents. "
    "Seeing the same few examples repeatedly can cause the agent to overfit to those examples.\n\n"
    "For example, if you ask an agent to process a batch of 20 resumes and one example in the prompt "
    "includes visiting a job description, the agent might end up visiting the same job description "
    "20 times for all resumes.\n\n"
    "Their proposed solution is to introduce small, structured variations each time an example is used, "
    "such as different phrasing, minor formatting changes, or small amounts of noise."
)

label = classify_text(text)
label

 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': '3. Dynamic few-shot prompting\n\nThey cautioned against using traditional few-shot prompting for agents. Seeing the same few examples repeatedly can cause the agent to overfit to those examples.\n\nFor example, if you ask an agent to process a batch of 20 resumes and one example in the prompt includes visiting a job description, the agent might end up visiting the same job description 20 times for all resumes.\n\nTheir proposed solution is to introduce small, structured variations each time an example is used, such as different phrasing, minor formatting changes, or small amounts of noise.', 'labels': ['Technical content about AI, data, machine learning, NLP, or software engineering implementation details such as methods, architectures, algorithms, or code-level discussion', 'Non-technical content such as opinions, social or political commentary, industry news, announcements, founder stories,

['Generative AI']

In [102]:
text = (
    "New course: Document AI: From OCR to Agentic Doc Extraction, built with LandingAI, "
    "where I am executive chairman, and taught by David Park and Andrea Kropp.\n\n"
    "Much of the world’s data is locked in PDFs, JPEGs, and other document formats. "
    "This short course shows how to build agentic workflows that process documents accurately "
    "by breaking them into parts, examining each piece carefully, and extracting information "
    "through multiple iterations.\n\n"
    "Traditional Optical Character Recognition (OCR) captures text but often loses important "
    "context such as table headers, chart captions, or column reading order. After exploring "
    "OCR’s limitations, the course introduces an Agentic Document Extraction (ADE) framework "
    "that treats document pages visually as images to parse and extract structured information.\n\n"
    "Skills covered include:\n"
    "- Building agents to convert unstructured files into structured Markdown, HTML, and JSON\n"
    "- Parsing complex data such as forms, handwriting, or equations using ADE\n"
    "- Mapping extracted information to named fields with a specified schema and bounding boxes "
    "for grounding and validation\n"
    "- Deploying retrieval-augmented generation (RAG) applications with event-driven document processing\n\n"
    "The course demonstrates how to process documents such as financial invoices, medical records, "
    "and academic papers using modern document AI techniques. "
    "Learn more at https://deeplearning.ai/short-courses/document-ai-from-ocr-to-agentic-doc-extraction"
)

label = classify_text(text)
label

 --> Classification technical rule bases
 --> Binary classification by zero shot
{'sequence': 'New course: Document AI: From OCR to Agentic Doc Extraction, built with LandingAI, where I am executive chairman, and taught by David Park and Andrea Kropp.\n\nMuch of the world’s data is locked in PDFs, JPEGs, and other document formats. This short course shows how to build agentic workflows that process documents accurately by breaking them into parts, examining each piece carefully, and extracting information through multiple iterations.\n\nTraditional Optical Character Recognition (OCR) captures text but often loses important context such as table headers, chart captions, or column reading order. After exploring OCR’s limitations, the course introduces an Agentic Document Extraction (ADE) framework that treats document pages visually as images to parse and extract structured information.\n\nSkills covered include:\n- Building agents to convert unstructured files into structured Markdown, 

['Machine Learning']

In [103]:
def pd_to_text(df, post_url):
    return df[df.post_url == post_url].content.values[0]

In [106]:
# df.post_url.values
for url in df.post_url.values:
    text = pd_to_text(df, url)
    print("----------------------------------------")
    print(text)
    print("Categorizes : ", classify_text(text))

----------------------------------------
New post: nanochat miniseries v1

The correct way to think about LLMs is that you are not optimizing for a single specific model but for a family models controlled by a single dial (the compute you wish to spend) to achieve monotonically better results. This allows you to do careful science of scaling laws and ultimately this is what gives you the confidence that when you pay for "the big run", the extrapolation will work and your money will be well spent. For the first public release of nanochat my focus was on end-to-end pipeline that runs the whole LLM pipeline with all of its stages. Now after YOLOing a few runs earlier, I'm coming back around to flesh out some of the parts that I sped through, starting of course with pretraining, which is both computationally heavy and critical as the foundation of intelligence and knowledge in these models.

After locally tuning some of the hyperparameters, I swept out a number of models fixing the FLOPs b

Categorizes :  ['Non-technical']
----------------------------------------
Another year of rapid AI advances has created more opportunities than ever for anyone — including those just entering the field — to build software. In fact, many companies just can’t find enough skilled AI talent. Every winter holiday, I spend some time learning and building, and I hope you will too. This helps me sharpen old skills and learn new ones, and it can help you grow your career in tech.

To be skilled at building AI systems, I recommend that you:
- Take AI courses
- Practice building AI systems
- (Optionally) read research papers

Let me share why each of these is important.

I’ve heard some developers advise others to just plunge into building things without worrying about learning. This is bad advice! Unless you’re already surrounded by a community of experienced AI developers, plunging into building without understanding the foundations of AI means you’ll risk reinventing the wheel or — more likely

 --> MLOps rule bases
 --> Zero shot tech classifcation
Categorizes :  ['Machine Learning', 'Generative AI']
----------------------------------------
Our Segment Anything Models are helping advance flood monitoring and disaster response.

See how 
@USRAedu
 and 
@USGS
 have fine-tuned SAM to automate a key bottleneck in real-time river mapping, enabling faster, scalable, and more cost-effective disaster preparedness: 
go.meta.me/9ec621
 --> Classification technical rule bases
 --> Binary classification by zero shot
Categorizes :  ['Non-technical']
----------------------------------------
We're thrilled to share the open source release of Meta Seal, a comprehensive, SOTA, and MIT-licensed suite of AI watermarking research, models, & training code.

Learn more in the  below and explore the artifacts here:
 --> Classification technical rule bases
 --> Binary classification by zero shot
 --> MLOps rule bases
 --> Zero shot tech classifcation
Categorizes :  ['Generative AI']
---------------

 --> MLOps rule bases
 --> Zero shot tech classifcation
Categorizes :  ['Generative AI', 'Machine Learning']
----------------------------------------
Sakana AI's geopolitical analyst Junya Ishii explained China's cognitive warfare against Japan in a Reuters article. By utilizing our proprietary algorithm "AB-MCTS," we have achieved structural analysis of complex narratives on SNS. As a Japanese AI company, we will prioritize national interests and contribute to the fields of security and diplomacy. 
@ReutersJapan
 --> Classification technical rule bases
 --> Binary classification by zero shot
Categorizes :  ['Non-technical']
----------------------------------------
Thinking a lot about the difference between these two images, and the fact that they're separated by only about 11 weeks
 --> Text is not long enough
Categorizes :  ['Non-technical']
----------------------------------------
Somewhere a quiet engineer closes their laptop.

They walk out before the sky finishes its warning.
 -

 --> MLOps rule bases
 --> Zero shot tech classifcation
Categorizes :  ['Generative AI']
----------------------------------------
New updated 2nd Edition, by 
@keithbourne
 via 
@PacktDataML
"Unlocking Data with Generative AI and RAG — Learn AI Agent Fundamentals with RAG-powered Memory, Graph-based RAG, and Intelligent Recall"

Get the book here: 
amzn.to/49zsIkb
 --> Classification technical rule bases
 --> Binary classification by zero shot
 --> MLOps rule bases
 --> Zero shot tech classifcation
Categorizes :  ['Generative AI']
----------------------------------------
New AI lab just dropped.

Brett Adcock (founder of Figure, the humanoid robotics company valued at $39B) is reportedly self-funding $100M into the new lab called "Hark."

The lab is building "human-centric AI" that can "think proactively, recursively improve and care deeply about people"
 --> Classification technical rule bases
Categorizes :  ['Non-technical']
----------------------------------------
Strategic choices 

...

Categorizes :  ['Non-technical']
----------------------------------------
Introducing ChatGPT Health — a dedicated space for health conversations in ChatGPT. You can securely connect medical records and wellness apps so responses are grounded in your own health information.

Designed to help you navigate medical care, not replace it. 

Join the waitlist to get early access.
 --> Classification technical rule bases
Categorizes :  ['Non-technical']
----------------------------------------
I've built a custom camera control 
@gradio
 component for camera control LoRAs for image models  

Here's a demo of 
@fal
's Qwen-Image-Edit-2511-Multiple-Angles-LoRA using the interactive camera component
 --> Classification technical rule bases
Categorizes :  ['Non-technical']
----------------------------------------
Introducing Qwen3-VL-Embedding and Qwen3-VL-Reranker – advancing the state of the art in multimodal retrieval and cross-modal understanding!
 Highlights:
 Built upon the robust Qwen3-VL 

 --> MLOps rule bases
 --> Zero shot tech classifcation
Categorizes :  ['Machine Learning']
----------------------------------------
This is an ad from General Motors (
@GM
) in 1969.

#ad #tech #AI #iot #FutureOfWork
 --> Text is not long enough
Categorizes :  ['Non-technical']
----------------------------------------
Today, we're joined by 
@rdn_nikita
, co-founder and CEO of 
@FlexionRobotics
 to discuss the gap between current robotic capabilities and what’s required to deploy fully autonomous robots in the real world. Nikita explains how reinforcement learning and simulation have driven rapid progress in robot locomotion—and why locomotion is still far from “solved.” We dig into the sim2real gap, and how adding visual inputs introduces noise and significantly complicates sim-to-real transfer. We also explore the debate between end-to-end models and modular approaches, and why separating locomotion, planning, and semantics remains a pragmatic approach today. Nikita also introduces 

Categorizes :  ['Non-technical']
----------------------------------------
The Future of AI Starts in the Ground

The future isn’t just being predicted—it’s being pulled from the earth and polished by AI & innovation. 

From the lithium in our batteries to the rare earth elements in our processors, the future of Tech and AI starts in the ground.

Saudi Arabia is staging a "global leap," jumping to 23rd in mining attractiveness. With $9.4T in minerals beneath the Arabian Shield, the #FutureMineralsPioneers are the architects of this new era.

I’m excited to see 70 elite teams from 30+ countries compete in Riyadh (Jan 8-10) to unlock this potential. This is how you diversify an economy! 

Join us at the Future Minerals Forum to see the winners crowned: 
zurl.co/KKkGB 

#FutureMineralsForum #NIDLP #Vision2030
 --> Classification technical rule bases
 --> Binary classification by zero shot
 --> MLOps rule bases
 --> Zero shot tech classifcation
Categorizes :  ['Generative AI']
-------------

Categorizes :  ['Non-technical']
----------------------------------------
Epoch 2
 --> Text is not long enough
Categorizes :  ['Non-technical']
----------------------------------------
I'm Boris and I created Claude Code. Lots of people have asked how I use Claude Code, so I wanted to show off my setup a bit.

My setup might be surprisingly vanilla! Claude Code works great out of the box, so I personally don't customize it much. There is no one correct way to use Claude Code: we intentionally build it in a way that you can use it, customize it, and hack it however you like. Each person on the Claude Code team uses it very differently.

So, here goes.
 --> Classification technical rule bases
Categorizes :  ['Non-technical']
----------------------------------------
The catch? Only works with LOCAL Chrome.

The native host has to run on the same machine as the browser. Claude Code expects to spawn a local process.

That's dangerous. A rogue website could do prompt injection to extract you

Categorizes :  ['Non-technical']
----------------------------------------
Getting old is everyone sending all the "Happy New Year" texts the next morning.
 --> Text is not long enough
Categorizes :  ['Non-technical']
----------------------------------------
Really good wrap up of the year in AI.
 --> Text is not long enough
Categorizes :  ['Non-technical']
----------------------------------------
One of my holiday rituals every year is to go down a rabbit hole of 
@BrandSanderson
 content and come away inspired by how awesome of a *human* he is.

Brandon is known for his variety of fantasy series set in his Cosmere universe but I originally discovered Brandon's work through an indirect route of wanting to try and write a book of fiction myself years ago. He teaches a course on writing at BYU and the videos for that are some of the best "how to write" content you'll find anywhere.

You'll quickly learn a few things about him

- the man is insanely productive. He has taken a natural gift